# Visualize the superimposed AF2 and AF3 structures (unrelaxed)

To understand if AF3 is sufficient

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import py3Dmol
import random
from glob import glob
import os
import json

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [2]:
def view_multiple(pdbs, n=9, cols=3):
    rows = (len(pdbs) + cols - 1) // cols
    
    view = py3Dmol.view(
        viewergrid=(rows, cols),
        width=300 * cols,
        height=300 * rows
    )
    for i, pdb_path in enumerate(pdbs):
        with open(pdb_path) as f:
            pdb_data = f.read()
        row, col = divmod(i, cols)
        view.addModel(pdb_data, "pdb", viewer=(row, col))
        
        # AF3 complex: MHC (chain B) and peptide (chain C) in grey
        view.setStyle(
            {"chain": "B"},
            {"cartoon": {"color": "lightgrey"}},
            viewer=(row, col)
        )
        view.setStyle(
            {"chain": "C"},
            {"stick": {"color": "lightgrey"}},
            viewer=(row, col)
        )
        # AF3 binder (chain A) in cornflowerblue
        view.setStyle(
            {"chain": "A"},
            {"cartoon": {"color": "cornflowerblue"}},
            viewer=(row, col)
        )
        # AF2 monomer (chain Z) in tomato red
        view.setStyle(
            {"chain": "Z"},
            {"cartoon": {"color": "tomato"}},
            viewer=(row, col)
        )
        
        # Structure name as label — stem of filename, positioned at origin
        name = os.path.splitext(os.path.basename(pdb_path))[0].split('_')[0]
        view.addLabel(
            name,
            {
                "position":        {"x": 0, "y": 0, "z": 0},
                "backgroundColor": "black",
                "backgroundOpacity": 0.6,
                "fontColor":       "white",
                "fontSize":        12,
                "fontOpacity":     1.0,
                "inFront":         True,
            },
            viewer=(row, col)
        )
        view.zoomTo(viewer=(row, col))
    return view.show()

In [3]:
pdb_list = glob('~/Desktop/pmhc_binder_kras/post_filter/outputs/author_design_stats/af2_af3_superimposed_pdb/*.pdb')
view_multiple(list(pdb_list))

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

-- RMSD summary (18/38 designs) --
  mean   : 0.927 A
  
  median : 0.735 A
  
  min    : 0.322 A  (hiv-10_20260412_181551)
  
  max    : 2.233 A  (mage-282_20260412_180303)
  
  std    : 0.562 A

Per design (sorted by RMSD):

    hiv-10_20260412_181551  0.322 A
    
    gp100-3_20260412_181553  0.366 A
    
    ctnnb1-15_20260412_181420  0.422 A
    
    mage-513_20260412_182022  0.456 A
    
    phox2b-5_20260412_182455  0.470 A
    
    wt1-5_20260412_182852  0.492 A
    
    pap-116_20260412_182455  0.645 A
    
    wt1-8_20260412_182853  0.674 A
    
    mart1-3_20260412_182022  0.709 A
    
    prame-9_20260415_002235  0.761 A
    
    phox2b-11_20260412_182455  0.834 A
    
    yfv-2_20260412_182851  1.015 A
    
    prame-2_20260413_024822  1.127 A
    
    mage-4_20260412_182023  1.360 A
    
    mart1-43_20260412_182022  1.380 A
    
    hiv-9_20260412_182024  1.403 A
    
    sars-6_20260415_002840  2.025 A
    
    mage-282_20260412_180303  2.233 A
    

## Conclusion:
1. For most designs, AF2 and AF3 binders look largely the same, except loopy regions
2. sars-6 and mage-282: the interface looks different (largely the same but the residues interacting may be different), but in sars-6, both have the loopy region in proximity to the peptide

## Check the per-residue pLDDT of the loop region (aa 25-31, 1-based) in sars-6

In [35]:
af2_json = "~/Desktop/pmhc_binder_kras/post_filter/outputs/r2/af2_monomer/author_sars-6/confidence_model_1_pred_0.json"
af3_json = "~/Desktop/pmhc_binder_kras/post_filter/outputs/author_design_stats/af3/sars-6_20260415_002840/seed-4_sample-3/confidences.json" # top ranking model for this design

with open(af2_json) as f:
    d = json.load(f)
    scores   = d['confidenceScore']
print(scores[24:31])

residue_atom_mapping = {
    25:[192,193,194,195,196],
    26:[197,198,199,200],
    27:[201, 202, 203, 204, 205, 206, 207, 208],
    28:[209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219],
    29:[220, 221, 222, 223, 224, 225, 226, 227],
    30:[228, 229, 230, 231, 232, 233, 234, 235, 236],
    31:[237, 238, 239, 240, 241, 242, 243]
                       } # all 1-based
new_residue_atom_mapping = {atom: res for res, atoms in residue_atom_mapping.items() for atom in atoms}
# print(new_residue_atom_mapping)

binder_chain = 'A'
with open(af3_json) as f:
    d = json.load(f)
    atom_plddts   = d['atom_plddts']
    atom_chain_ids = d['atom_chain_ids']
    binder_atoms = [
        (i, pldt) for i, (pldt, ch) in enumerate(zip(atom_plddts, atom_chain_ids))
        if ch == binder_chain
    ]
#     print(binder_atoms)
    residue_plddt_tmp = dict()
    for residue_index in residue_atom_mapping:
        residue_plddt_tmp[residue_index] = []
    for atom_index, plddt in binder_atoms:
        atom_index += 1
        if atom_index in new_residue_atom_mapping:
            residue_index = new_residue_atom_mapping[atom_index]
            residue_plddt_tmp[residue_index].append(plddt)
#     print(residue_plddt_tmp)

residue_plddt = dict()
for res, plddt_list in residue_plddt_tmp.items():
    residue_plddt[res] = round(sum(plddt_list)/len(plddt_list), 2)
print(residue_plddt)

[89.42, 89.34, 88.09, 89.0, 90.08, 92.6, 93.23]
{25: 94.08, 26: 94.71, 27: 95.19, 28: 84.86, 29: 92.44, 30: 86.26, 31: 92.76}


## Conclusion:
The loopy region has high confidence (not disordered) and the binding is real. In this case, loopy contact with the peptide may be favorable over alpha helices.

For future KRAS G12D designs, if seeing high pLDDT in loops, if they contact p5 in pMHC fold structure, these designs should be kept